# Tokenization

[视频](https://www.youtube.com/watch?v=zduSFxRajkE)<br>
[代码仓库](https://github.com/karpathy/minbpe)<br>
[Eureka Labs Discord](https://discord.com/invite/3zy8kqD9Cp)

## 目录

- [回顾：字符级 Tokenization](#回顾字符级-tokenization)
- [Tokenization 的陷阱](#分词的陷阱)
- [tokenizer 的核心思想](#tokenizer-的思想)
  - [Unicode](#unicode)
- [Byte-Pair Encoding](#字节对编码)
  - [解码](#解码)
  - [编码](#编码)
- [迈向 SOTA：实际应用中的 tokenizer](#走向-sota实战中的-tokenizer)
  - [GPT tokenizer](#gpt-tokenizer)
  - [OpenAI TikToken](#openai-tiktoken)
  - [特殊 token](#特殊-token)
  - [Sentencepiece](#sentencepiece)
- [练习时间](#练习时间)
- [回过头看：vocab_size](#回过头来vocab_size)
- [总结](#结论)
- [参考文献](#参考文献)

<style>
/* Keep notebook content printable without horizontal clipping. */
.jp-OutputArea-output img,
.jp-RenderedImage img,
img {
  max-width: 100% !important;
  height: auto !important;
}

.jp-Cell,
.jp-InputArea,
.jp-OutputArea-output,
.jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}

.jp-InputArea-editor pre,
.jp-RenderedText pre,
.jp-OutputArea-output pre,
.jp-OutputArea-output code,
.highlight pre,
.input_area pre,
.output_area pre,
.output_subarea pre,
.output_text pre,
.output_stream pre,
pre,
code {
  white-space: pre-wrap !important;
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}

.rendered_html table,
.jp-RenderedHTMLCommon table {
  max-width: 100% !important;
}
</style>


理解大型语言模型（LLMs）的内部运作机制，需要深入探究生成过程所需的第一步：**Tokenization**。<br><br>
**Tokenization 描述了将字符序列转换为一系列具有数值代表性的 token 的过程。**<br>
**token 是语言模型的基本语义单位。**<br><br>
LLMs 是*纯数学*模型。它们无法直接处理原始文本，而是*只处理数字*。<br>
这个任务看似简单：找到一种最合适的方式，将输入文本映射为可处理的数值表示。<br><br>
如果这一映射处理不当，该转换往往会成为众多下游问题的*根本*原因。

## 回顾：字符级 Tokenization

在[上一讲](../N007%20-%20GPT%20From%20Scratch/N007%20-%20GPT.ipynb)中，我们已经为 GPT 模型实现了一种简化形式的 tokenization。<br>
下面快速回顾一下那个过程：

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import regex as re
import tiktoken
import os
import json

首先，我们从 `tiny-shakespeare.txt` 文本文件中加载了一个示例数据集到内存中：

In [3]:
# Read the txt file to inspect it
with open('../tiny-shakespeare.txt', 'r') as f:
    text = f.read()

# Print a sample of the text (First 100 characters)
print("Length of Dataset:", len(text), "\n")
print(text[:100])

Length of Dataset: 1115394 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


然后，我们确定了数据集中存在的所有唯一字符：

In [3]:
chars = sorted(list(set(text))) # Get all unique characters in the text
vocab_size = len(chars)         # Length of vocabulary (this includes the space character and newline)
print('Unique Characters:', ''.join(chars))
print(f'\nVocabulary size: {vocab_size}')

Unique Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Vocabulary size: 65


鉴于在文本文件中找到的这些唯一字符，我们接着将每个字符映射到相应的整数，<br>
从而实现字符乃至整个数据集的完全数值表示：

In [4]:
stoi = { ch:i for i,ch in enumerate(chars) }     # Character to index mapping
itos = { i:ch for i,ch in enumerate(chars) }     # Index to character mapping

# This is the encoder; Encode a string to a list of integers
encode = lambda s: [stoi[c] for c in s]
# This is the decoder; Decode a list of integers to a string
decode = lambda l: ''.join([itos[i] for i in l])

msg = "hii there"
token_list = encode(msg)
print(token_list)
print(decode(token_list))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


> 上述实现的方法被称为**字符级 tokenization**。

对输入文本文件中所有唯一字符进行编码，将得到同等数量的 token 供我们的 GPT 处理。<br>
但是，我们并没有就此止步。<br>
相反，我们接着构建了一个查找表，将每个数值 token 映射到一个向量表示。<br>
<br>
**我们为什么要多走这一步？**<br>
由 $65$ 个向量（每个 token 一个），每个 $65$ 维（同样每个 token 一维）构成的词表表示，为每个 token 提供了一种更深层的、固定分辨率的表示。<br>
然后，我们让 `BigramLM` 针对每个 token，去优化该 token 向量表示内部的值。<br>
<br>
**总而言之，用可学习向量对 token 进行统一大小的表示，使 `BigramLM` 能够更好地捕捉字符之间的组合关系。**<br><br>
**可以这样理解：** 在语义或功能上相似的字符，现在可以获得相似的向量表示。<br>
因此，元音或辅音可能会在向量空间中聚集在一起，这使得 `BigramLM` 更容易捕捉关联中的相似性。<br>
向量比基于离散整数的 token 表示要富有表现力得多。<br><br>实现这一切的 Embedding 层 `BigramLM` 如下所示：

In [5]:
torch.manual_seed(1337) # for reproducibility

# Not really an LM at this stage, but we will get there...
class BigramLM(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Embedding the vocabulary
        # Every one of the vocab_size tokens is represented by a vector of size vocab_size
        self.embed = nn.Embedding(vocab_size, vocab_size) # 65 unique 65-dim vectors

    def forward(self, idx, targets):
        # idx is of shape (batch_size, block_size)
        # targets is of shape (batch_size, block_size)
        logits = self.embed(idx)
        return logits # Embed the input indices, shape is now (batch_size, block_size, vocab_size) (B, T, C)

即便有了上述的 $65 \times 65$ embedding 矩阵，事实证明我们充其量只实现了一种低效的 tokenization。<br>
实际上，token 词表的构建方式截然不同，远不止字符级以及直接的"字符到数值再到向量"的表示映射。

> 实际上，文本并不是在字符级上进行 tokenization 的，而是在所谓的*chunk 级*上进行的。

**我们的目标是找到并探索一种在 chunk 级而非字符级上对文本进行 tokenization 的方法。**

## 分词的陷阱

让我们继续把 token 作为信息的基本单位这一基本概念。<br>
我们的目标是将文本字符串转换为对 LLM 而言最具信息量、最易于解释的表示形式。<br>
<br>
分词如果处理不当，可能会给 LLM 带来*大量*的下游问题。<br>
一些与分词相关的最常见问题包括：

- 为什么 LLM 无法正确拼写？
- 为什么 LLM 无法完成像反转字符串这样简单的文本操作？
- 为什么 LLM 在非英语语言上的表现可能更差？
- 为什么 LLM 无法正确执行简单的算术运算？
- 为什么输入某些特殊字符（如 `<|endoftext|>`）会导致生成停止？
- 为什么 LLM 会因为“尾部空格”而出现异常？
- 为什么 LLM 在遇到单词中出现大写字母时会出问题？
- 为什么通过 YAML 与 LLM 交互可能比使用 JSON 更可取？
- 为什么“LLM”实际上并不等同于“端到端语言建模”？

> 一切*罪恶*的根源是什么？**分词！**

让我们到 [tiktokenizer.vercel.app](https://tiktokenizer.vercel.app) 上用 [GPT-2 Tokenizer](https://insightcivic.s3.us-east-1.amazonaws.com/language-models.pdf) 看一个实际例子：

```md
Tokenization is at the heart of much weirdness of LLMs. Do not brush it off.

127 + 677 = 804
1275 + 6773 = 8041

Egg.
I have an Egg.
egg.
EGG.

만나서 반가워요. 저는 OpenAI에서 개발한 대규모 언어 모델인 ChatGPT입니다. 궁금한 것이 있으시면 무엇이든 물어보세요.

for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)
```
<br>

![](./img/Tiktoken_Vercel_1.png)

每种颜色代表一个不同的 token。很明显，GPT-2 的 tokenizer 并不是在字符级别上操作的。<br>
<br>
这在处理纯文本时可能影响不大，但请看看算术运算的分词结果。<br>
除了 $127$ 之外的每一个数值都被切分成了多个 token。<br>
这种按 token 的逻辑性错误切分在这里可能会有问题，因为它可能导致难以保留任何数值或算术表达式的语义含义。<br>
<br>
有趣的是，“Egg”这个词根据大小写程度和前导空格的不同，被分成了四种不同的方式。<br>
我们还可以看到，韩语文本的分词比英语文本更加细粒度。<br>
这在很大程度上可以归因于训练数据中的表示不平衡。tokenizer 可能只是更针对英语文本进行了优化。<br>
正因如此，韩语使用者在使用 LLM 完成相同任务时，可能需要比英语使用者向 LLM 提供商支付更高的费用，这仅仅是因为更差/更细粒度的分词会更快地填满上下文窗口的 token 计数。<br>
此外，用更多 token 来表示相同数量的数据，会导致下游的注意力缓冲区更快被填满。<br>
这不可避免地会导致 LLM 整体性能的下降。<br>
<br>
在 Python 代码示例中，每个空格都被单独分词。这降低了 LLM 对代码的可解释性，因为上下文窗口会迅速被空格 token 充斥。<br>
就像韩语文本的情况一样，这也会对模型性能产生负面影响。

当向 GPT-4 tokenizer（即 `cl100k_base`）提供相同的输入时，我们的 token 数量几乎减半。<br>
这意味着该 tokenizer 的词表大小约为 GPT-2 tokenizer 的两倍：

![](./img/cl100k_base_1.png)

> 仅仅是词表大小的增加，*就*使得 GPT-4 的上下文窗口大小几乎比 GPT-2 翻了一番。

## Tokenizer 的思想

> tokenizer 是一个独立于 LLM 的实体。<br>它可以独立地在特定文本上进行调优或训练，而 LLM 则基于不同的文本进行优化。<br>然而，关键的是，虽然 tokenizer 不依赖于 LLM，但它充当了 LLM 与文本之间的接口。<br>因此，LLM 得以在一个纯 token 化的世界中运行。

![](./img/Tokenizer_Schema.png)

现在我们想要创建一个超越字符级分词的 Tokenizer，用某个标识值来表示某些文本块。<br>
在此基础上，我们可以继续构建一个查找表，将每个表示文本块的 token 映射到一个向量表示。这一步与之前一样。

哦，还有，别忘了，这次 tokenizer 还应该能够处理不同的文字系统。还有 Emoji，*我们需要 Emoji 支持！*

### Unicode

Python 开箱即用即可处理不同的文字系统：

In [4]:
some_text = "안녕하세요 👋 (hello in Korean!)"
print(some_text)

안녕하세요 👋 (hello in Korean!)


[文档](https://docs.python.org/3/library/stdtypes.html#text-sequence-type-str) 中指出，Python 的 `str` 类型是一个 **[Unicode](https://en.wikipedia.org/wiki/Unicode) 码点序列**。<br>
Unicode 是一种字符编码标准，它将每个字符映射到一个唯一的整数值。<br><br>
我们可以使用 `ord()` 函数来读取这个值：

In [5]:
print([ord(x) for x in some_text])

[50504, 45397, 54616, 49464, 50836, 32, 128075, 32, 40, 104, 101, 108, 108, 111, 32, 105, 110, 32, 75, 111, 114, 101, 97, 110, 33, 41]


就这样,我们解决了编码问题,……对吧?<br>
**嗯,并非如此。**<br>
<br>
虽然 Unicode 内容全面,但它一直在变化、演进,而且已经相当庞大。**Unicode 并不稳定。**<br>
但是,Unicode 定义了三种 *稳定的* 编码形式,可以用来表示 Unicode 字符的一个子集:[UTF-8](https://en.wikipedia.org/wiki/UTF-8)、[UTF-16](https://en.wikipedia.org/wiki/UTF-16) 和 [UTF-32](https://en.wikipedia.org/wiki/UTF-32)。<br>
通过这些编码形式来表示字符,是借助字节序列实现的。<br>

> **为什么从上面的代码可以看到 Unicode 用整数来表示字符,而 UTF-8、UTF-16 和 UTF-32 却使用字节?**<br>
> UTF-8、UTF-16 和 UTF-32 是编码方案,定义了 Unicode 如何以二进制形式(字节序列)进行存储和传输。<br>
> 例如,UTF-8 专门用于在自身保持稳定的同时,应对 Unicode 标准不断演进的特性。<br>
> Unicode 为字符分配整数值,而 UTF-8、UTF-16 和 UTF-32 是底层编码方案,描述了这些码点如何被转换为字节序列以进行存储和传输。

使用 UTF-8,你可以确定地用 $1$ 到 $4$ 个字节的序列来表示 Unicode 字符。**这是固定的。**<br>
你可以参考 [Nathan Reed 关于 Unicode 的博客文章](https://www.reedbeta.com/blog/programmers-intro-to-unicode/) 了解更多细节,尤其是 [UTF-8 Everywhere 宣言](https://utf8everywhere.org/)。

以下是根据 UTF-8 表示我们字符串的原始字节集合:

In [7]:
list(some_text.encode('utf-8'))

[236,
 149,
 136,
 235,
 133,
 149,
 237,
 149,
...
 41]


尽管如此,我们并不想直接把原始字节当作 token 使用。**但为什么呢?**<br>
<br>
从上面可以看到,我们只能区分 $256$ 种不同的表示,因为我们使用的是 $8$ 位表示/字节。<br>
虽然我们确实可以用这种逐字节的表示来表示字节数组,但这是一种非常低效的文本表示方式。<br>
这本质上相当于把文本放大,在一个非常细的粒度上区分它的各个部分(字母编码的字节部分),从而不可避免地拉长了 token 数量。<br>
而这反过来会塞满 LLM 的注意力缓冲区,限制其下游能力。

> 我们希望支持一个较大的词表,以 UTF-8 作为分词(tokenization)的基础,但 **我们不想直接用原始字节作为 token,因为那样效率太低**。

(我们应该注意,从这里开始,我们所做的任何事情都会在 UTF-8 之上引入某种开销或抽象。在理想世界中,这应当被避免。有趣的是,像 [\[Yu et al., 2023\]](https://arxiv.org/abs/2305.07185) 这样的工作正是在尝试这样做:避免分词开销,更接近于直接使用原始字节作为 token。不过,其证明尚待完成。)

## 字节对编码

[维基百科上关于字节对编码 (BPE) 的词条](https://en.wikipedia.org/wiki/Byte_pair_encoding) 非常实用,强烈推荐阅读。<br>

> 迭代地,对于给定的文本,BPE 将出现最频繁的连续 token 对合并为一个新的单一 token,然后我们把它加入词表。<br> 我们重复这一过程,直到达到预定义的词表大小。<br>

你可以看到这如何构建出 token 的层级结构:最频繁的对先被合并,然后是次频繁的对,依此类推。<br>
<br>
**让我们看看这在实践中是如何运作的:**

In [8]:
# Text is the first paragraph from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode('utf-8')   # Byte array
tokens = list(map(int, tokens)) # Convert to list of integers for visualization

print(text, "\n")
print("Original Text Length:", len(text), "\n\n")
print(tokens, "\n")
print("Length of Token List:", len(tokens)) # Simple characters map to one byte, but e.g. emojis map to 4 bytes

Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception. 

Original Text Length: 533 


[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 22

In [9]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]): # Sliding Window of size 2 across tokens 
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
print(sorted(((v,k) for k,v in stats.items()), reverse=True)) # Invert key-value relationship and sort by key (count)
print(sorted(((v,(chr(k[0]), chr(k[1]))) for k,v in stats.items()), reverse=True)[:5]) # Just for fun, 5 most common bigrams written out

top_pair = max(stats, key=stats.get) # Retrieve the most common bigram
print(top_pair)

[(20, (101, 32)), (15, (240, 159)), (12, (226, 128)), (12, (105, 110)), (10, (115, 32)), (10, (97, 110)), (10, (32, 97)), (9, (32, 116)), (8, (116, 104)), (7, (159, 135)), (7, (159, 133)), (7, (97, 114)), (6, (239, 189)), (6, (140, 240)), (6, (128, 140)), (6, (116, 32)), (6, (114, 32)), (6, (111, 114)), (6, (110, 103)), (6, (110, 100)), (6, (109, 101)), (6, (104, 101)), (6, (101, 114)), (6, (32, 105)), (5, (117, 115)), (5, (115, 116)), (5, (110, 32)), (5, (100, 101)), (5, (44, 32)), (5, (32, 115)), (4, (116, 105)), (4, (116, 101)), (4, (115, 44)), (4, (114, 105)), (4, (111, 117)), (4, (111, 100)), (4, (110, 116)), (4, (110, 105)), (4, (105, 99)), (4, (104, 97)), (4, (103, 32)), (4, (101, 97)), (4, (100, 32)), (4, (99, 111)), (4, (97, 109)), (4, (85, 110)), (4, (32, 119)), (4, (32, 111)), (4, (32, 102)), (4, (32, 85)), (3, (118, 101)), (3, (116, 115)), (3, (116, 114)), (3, (116, 111)), (3, (114, 116)), (3, (114, 115)), (3, (114, 101)), (3, (111, 102)), (3, (111, 32)), (3, (108, 108)), (

此时 token 词表的范围是 $0$ 到 $255$,因为我们使用的是 $8$ 位表示/字节。<br>
我们现在可以创建一个新的、第 $256^{\text{th}}$ 个 token,用来表示最常见的对 `('e', ' ')`:

In [10]:
def merge(ids, pair, idx):
    # Iterating through ids, if we find (pair), replace it by value idx
    newids = []
    i = 0
    while i < len(ids):
        # If we are not at the very last position AND the pair matches, replace it
        if i < len(ids)-1 and (ids[i], ids[i+1]) == pair:
            newids.append(idx)
            i += 2 # We skip over the now replaced pair
        else:
            newids.append(ids[i])
            i += 1
    return newids

# Sanity-Check
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

[5, 6, 99, 9, 1]


In [11]:
tokens2 = merge(tokens, top_pair, 256)

print(tokens2, "\n")
print("Length of Token List:", len(tokens2))
print("All Occurrences Removed" if (True if top_pair not in zip(tokens2, tokens2[1:]) else False) else "Still Some Occurrences Left")

[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 256, 118, 101, 114, 121, 32, 110, 97, 109, 256, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 256, 105, 110, 116, 111, 32, 116, 104, 256, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 256, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 256, 111, 117, 103, 104, 116, 32, 116, 111, 32, 226, 128, 156

我们实际上把文本映射到了 UTF-8,找出了最常见的连续字节值对(在 $8$ 位取值范围内为 $0$ 到 $255$),并用一个新 token($256$)替换了它的出现位置,该 token 表示对应的字节对。**这描述了我们到目前为止所做的事情。**<br>
不过,我们还没有记录下我们的操作。

既然我们已经确认基础实现可以工作,就可以根据需要循环遍历 token 多少次都行,真正构建出一个词表。<br>
你可以把这种词表扩充过程想象成从叶子节点向下迭代地构建一棵二叉树,直到根节点。<br>
<br>
我们将使用整篇博客文章作为训练数据:

In [12]:
text = """A Programmer’s Introduction to Unicode March 3, 2017 · Coding · 22 Comments  Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺\u200c🇳\u200c🇮\u200c🇨\u200c🇴\u200c🇩\u200c🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.  A few months ago, I got interested in Unicode and decided to spend some time learning more about it in detail. In this article, I’ll give an introduction to it from a programmer’s point of view.  I’m going to focus on the character set and what’s involved in working with strings and files of Unicode text. However, in this article I’m not going to talk about fonts, text layout/shaping/rendering, or localization in detail—those are separate issues, beyond my scope (and knowledge) here.  Diversity and Inherent Complexity The Unicode Codespace Codespace Allocation Scripts Usage Frequency Encodings UTF-8 UTF-16 Combining Marks Canonical Equivalence Normalization Forms Grapheme Clusters And More… Diversity and Inherent Complexity As soon as you start to study Unicode, it becomes clear that it represents a large jump in complexity over character sets like ASCII that you may be more familiar with. It’s not just that Unicode contains a much larger number of characters, although that’s part of it. Unicode also has a great deal of internal structure, features, and special cases, making it much more than what one might expect a mere “character set” to be. We’ll see some of that later in this article.  When confronting all this complexity, especially as an engineer, it’s hard not to find oneself asking, “Why do we need all this? Is this really necessary? Couldn’t it be simplified?”  However, Unicode aims to faithfully represent the entire world’s writing systems. The Unicode Consortium’s stated goal is “enabling people around the world to use computers in any language”. And as you might imagine, the diversity of written languages is immense! To date, Unicode supports 135 different scripts, covering some 1100 languages, and there’s still a long tail of over 100 unsupported scripts, both modern and historical, which people are still working to add.  Given this enormous diversity, it’s inevitable that representing it is a complicated project. Unicode embraces that diversity, and accepts the complexity inherent in its mission to include all human writing systems. It doesn’t make a lot of trade-offs in the name of simplification, and it makes exceptions to its own rules where necessary to further its mission.  Moreover, Unicode is committed not just to supporting texts in any single language, but also to letting multiple languages coexist within one text—which introduces even more complexity.  Most programming languages have libraries available to handle the gory low-level details of text manipulation, but as a programmer, you’ll still need to know about certain Unicode features in order to know when and how to apply them. It may take some time to wrap your head around it all, but don’t be discouraged—think about the billions of people for whom your software will be more accessible through supporting text in their language. Embrace the complexity!  The Unicode Codespace Let’s start with some general orientation. The basic elements of Unicode—its “characters”, although that term isn’t quite right—are called code points. Code points are identified by number, customarily written in hexadecimal with the prefix “U+”, such as U+0041 “A” latin capital letter a or U+03B8 “θ” greek small letter theta. Each code point also has a short name, and quite a few other properties, specified in the Unicode Character Database.  The set of all possible code points is called the codespace. The Unicode codespace consists of 1,114,112 code points. However, only 128,237 of them—about 12% of the codespace—are actually assigned, to date. There’s plenty of room for growth! Unicode also reserves an additional 137,468 code points as “private use” areas, which have no standardized meaning and are available for individual applications to define for their own purposes.  Codespace Allocation To get a feel for how the codespace is laid out, it’s helpful to visualize it. Below is a map of the entire codespace, with one pixel per code point. It’s arranged in tiles for visual coherence; each small square is 16×16 = 256 code points, and each large square is a “plane” of 65,536 code points. There are 17 planes altogether.  Map of the Unicode codespace (click to zoom)  White represents unassigned space. Blue is assigned code points, green is private-use areas, and the small red area is surrogates (more about those later). As you can see, the assigned code points are distributed somewhat sparsely, but concentrated in the first three planes.  Plane 0 is also known as the “Basic Multilingual Plane”, or BMP. The BMP contains essentially all the characters needed for modern text in any script, including Latin, Cyrillic, Greek, Han (Chinese), Japanese, Korean, Arabic, Hebrew, Devanagari (Indian), and many more.  (In the past, the codespace was just the BMP and no more—Unicode was originally conceived as a straightforward 16-bit encoding, with only 65,536 code points. It was expanded to its current size in 1996. However, the vast majority of code points in modern text belong to the BMP.)  Plane 1 contains historical scripts, such as Sumerian cuneiform and Egyptian hieroglyphs, as well as emoji and various other symbols. Plane 2 contains a large block of less-common and historical Han characters. The remaining planes are empty, except for a small number of rarely-used formatting characters in Plane 14; planes 15–16 are reserved entirely for private use.  Scripts Let’s zoom in on the first three planes, since that’s where the action is:  Map of scripts in Unicode planes 0–2 (click to zoom)  This map color-codes the 135 different scripts in Unicode. You can see how Han () and Korean () take up most of the range of the BMP (the left large square). By contrast, all of the European, Middle Eastern, and South Asian scripts fit into the first row of the BMP in this diagram.  Many areas of the codespace are adapted or copied from earlier encodings. For example, the first 128 code points of Unicode are just a copy of ASCII. This has clear benefits for compatibility—it’s easy to losslessly convert texts from smaller encodings into Unicode (and the other direction too, as long as no characters outside the smaller encoding are used).  Usage Frequency One more interesting way to visualize the codespace is to look at the distribution of usage—in other words, how often each code point is actually used in real-world texts. Below is a heat map of planes 0–2 based on a large sample of text from Wikipedia and Twitter (all languages). Frequency increases from black (never seen) through red and yellow to white.  Heat map of code point usage frequency in Unicode planes 0–2 (click to zoom)  You can see that the vast majority of this text sample lies in the BMP, with only scattered usage of code points from planes 1–2. The biggest exception is emoji, which show up here as the several bright squares in the bottom row of plane 1.  Encodings We’ve seen that Unicode code points are abstractly identified by their index in the codespace, ranging from U+0000 to U+10FFFF. But how do code points get represented as bytes, in memory or in a file?  The most convenient, computer-friendliest (and programmer-friendliest) thing to do would be to just store the code point index as a 32-bit integer. This works, but it consumes 4 bytes per code point, which is sort of a lot. Using 32-bit ints for Unicode will cost you a bunch of extra storage, memory, and performance in bandwidth-bound scenarios, if you work with a lot of text.  Consequently, there are several more-compact encodings for Unicode. The 32-bit integer encoding is officially called UTF-32 (UTF = “Unicode Transformation Format”), but it’s rarely used for storage. At most, it comes up sometimes as a temporary internal representation, for examining or operating on the code points in a string.  Much more commonly, you’ll see Unicode text encoded as either UTF-8 or UTF-16. These are both variable-length encodings, made up of 8-bit or 16-bit units, respectively. In these schemes, code points with smaller index values take up fewer bytes, which saves a lot of memory for typical texts. The trade-off is that processing UTF-8/16 texts is more programmatically involved, and likely slower.  UTF-8 In UTF-8, each code point is stored using 1 to 4 bytes, based on its index value.  UTF-8 uses a system of binary prefixes, in which the high bits of each byte mark whether it’s a single byte, the beginning of a multi-byte sequence, or a continuation byte; the remaining bits, concatenated, give the code point index. This table shows how it works:  UTF-8 (binary)\tCode point (binary)\tRange 0xxxxxxx\txxxxxxx\tU+0000–U+007F 110xxxxx 10yyyyyy\txxxxxyyyyyy\tU+0080–U+07FF 1110xxxx 10yyyyyy 10zzzzzz\txxxxyyyyyyzzzzzz\tU+0800–U+FFFF 11110xxx 10yyyyyy 10zzzzzz 10wwwwww\txxxyyyyyyzzzzzzwwwwww\tU+10000–U+10FFFF A handy property of UTF-8 is that code points below 128 (ASCII characters) are encoded as single bytes, and all non-ASCII code points are encoded using sequences of bytes 128–255. This has a couple of nice consequences. First, any strings or files out there that are already in ASCII can also be interpreted as UTF-8 without any conversion. Second, lots of widely-used string programming idioms—such as null termination, or delimiters (newlines, tabs, commas, slashes, etc.)—will just work on UTF-8 strings. ASCII bytes never occur inside the encoding of non-ASCII code points, so searching byte-wise for a null terminator or a delimiter will do the right thing.  Thanks to this convenience, it’s relatively simple to extend legacy ASCII programs and APIs to handle UTF-8 strings. UTF-8 is very widely used in the Unix/Linux and Web worlds, and many programmers argue UTF-8 should be the default encoding everywhere.  However, UTF-8 isn’t a drop-in replacement for ASCII strings in all respects. For instance, code that iterates over the “characters” in a string will need to decode UTF-8 and iterate over code points (or maybe grapheme clusters—more about those later), not bytes. When you measure the “length” of a string, you’ll need to think about whether you want the length in bytes, the length in code points, the width of the text when rendered, or something else.  UTF-16 The other encoding that you’re likely to encounter is UTF-16. It uses 16-bit words, with each code point stored as either 1 or 2 words.  Like UTF-8, we can express the UTF-16 encoding rules in the form of binary prefixes:  UTF-16 (binary)\tCode point (binary)\tRange xxxxxxxxxxxxxxxx\txxxxxxxxxxxxxxxx\tU+0000–U+FFFF 110110xxxxxxxxxx 110111yyyyyyyyyy\txxxxxxxxxxyyyyyyyyyy + 0x10000\tU+10000–U+10FFFF A more common way that people talk about UTF-16 encoding, though, is in terms of code points called “surrogates”. All the code points in the range U+D800–U+DFFF—or in other words, the code points that match the binary prefixes 110110 and 110111 in the table above—are reserved specifically for UTF-16 encoding, and don’t represent any valid characters on their own. They’re only meant to occur in the 2-word encoding pattern above, which is called a “surrogate pair”. Surrogate code points are illegal in any other context! They’re not allowed in UTF-8 or UTF-32 at all.  Historically, UTF-16 is a descendant of the original, pre-1996 versions of Unicode, in which there were only 65,536 code points. The original intention was that there would be no different “encodings”; Unicode was supposed to be a straightforward 16-bit character set. Later, the codespace was expanded to make room for a long tail of less-common (but still important) Han characters, which the Unicode designers didn’t originally plan for. Surrogates were then introduced, as—to put it bluntly—a kludge, allowing 16-bit encodings to access the new code points.  Today, Javascript uses UTF-16 as its standard string representation: if you ask for the length of a string, or iterate over it, etc., the result will be in UTF-16 words, with any code points outside the BMP expressed as surrogate pairs. UTF-16 is also used by the Microsoft Win32 APIs; though Win32 supports either 8-bit or 16-bit strings, the 8-bit version unaccountably still doesn’t support UTF-8—only legacy code-page encodings, like ANSI. This leaves UTF-16 as the only way to get proper Unicode support in Windows. (Update: in Win10 version 1903, they finally added UTF-8 support to the 8-bit APIs! 😊)  By the way, UTF-16’s words can be stored either little-endian or big-endian. Unicode has no opinion on that issue, though it does encourage the convention of putting U+FEFF zero width no-break space at the top of a UTF-16 file as a byte-order mark, to disambiguate the endianness. (If the file doesn’t match the system’s endianness, the BOM will be decoded as U+FFFE, which isn’t a valid code point.)  Combining Marks In the story so far, we’ve been focusing on code points. But in Unicode, a “character” can be more complicated than just an individual code point!  Unicode includes a system for dynamically composing characters, by combining multiple code points together. This is used in various ways to gain flexibility without causing a huge combinatorial explosion in the number of code points.  In European languages, for example, this shows up in the application of diacritics to letters. Unicode supports a wide range of diacritics, including acute and grave accents, umlauts, cedillas, and many more. All these diacritics can be applied to any letter of any alphabet—and in fact, multiple diacritics can be used on a single letter.  If Unicode tried to assign a distinct code point to every possible combination of letter and diacritics, things would rapidly get out of hand. Instead, the dynamic composition system enables you to construct the character you want, by starting with a base code point (the letter) and appending additional code points, called “combining marks”, to specify the diacritics. When a text renderer sees a sequence like this in a string, it automatically stacks the diacritics over or under the base letter to create a composed character.  For example, the accented character “Á” can be expressed as a string of two code points: U+0041 “A” latin capital letter a plus U+0301 “◌́” combining acute accent. This string automatically gets rendered as a single character: “Á”.  Now, Unicode does also include many “precomposed” code points, each representing a letter with some combination of diacritics already applied, such as U+00C1 “Á” latin capital letter a with acute or U+1EC7 “ệ” latin small letter e with circumflex and dot below. I suspect these are mostly inherited from older encodings that were assimilated into Unicode, and kept around for compatibility. In practice, there are precomposed code points for most of the common letter-with-diacritic combinations in European-script languages, so they don’t use dynamic composition that much in typical text.  Still, the system of combining marks does allow for an arbitrary number of diacritics to be stacked on any base character. The reductio-ad-absurdum of this is Zalgo text, which works by ͖͟ͅr͞aṋ̫̠̖͈̗d͖̻̹óm̪͙͕̗̝ļ͇̰͓̳̫ý͓̥̟͍ ̕s̫t̫̱͕̗̰̼̘͜a̼̩͖͇̠͈̣͝c̙͍k̖̱̹͍͘i̢n̨̺̝͇͇̟͙ģ̫̮͎̻̟ͅ ̕n̼̺͈͞u̮͙m̺̭̟̗͞e̞͓̰̤͓̫r̵o̖ṷs҉̪͍̭̬̝̤ ̮͉̝̞̗̟͠d̴̟̜̱͕͚i͇̫̼̯̭̜͡ḁ͙̻̼c̲̲̹r̨̠̹̣̰̦i̱t̤̻̤͍͙̘̕i̵̜̭̤̱͎c̵s ͘o̱̲͈̙͖͇̲͢n͘ ̜͈e̬̲̠̩ac͕̺̠͉h̷̪ ̺̣͖̱ḻ̫̬̝̹ḙ̙̺͙̭͓̲t̞̞͇̲͉͍t̷͔̪͉̲̻̠͙e̦̻͈͉͇r͇̭̭̬͖,̖́ ̜͙͓̣̭s̘̘͈o̱̰̤̲ͅ ̛̬̜̙t̼̦͕̱̹͕̥h̳̲͈͝ͅa̦t̻̲ ̻̟̭̦̖t̛̰̩h̠͕̳̝̫͕e͈̤̘͖̞͘y҉̝͙ ̷͉͔̰̠o̞̰v͈͈̳̘͜er̶f̰͈͔ḻ͕̘̫̺̲o̲̭͙͠ͅw̱̳̺ ͜t̸h͇̭͕̳͍e̖̯̟̠ ͍̞̜͔̩̪͜ļ͎̪̲͚i̝̲̹̙̩̹n̨̦̩̖ḙ̼̲̼͢ͅ ̬͝s̼͚̘̞͝p͙̘̻a̙c҉͉̜̤͈̯̖i̥͡n̦̠̱͟g̸̗̻̦̭̮̟ͅ ̳̪̠͖̳̯̕a̫͜n͝d͡ ̣̦̙ͅc̪̗r̴͙̮̦̹̳e͇͚̞͔̹̫͟a̙̺̙ț͔͎̘̹ͅe̥̩͍ a͖̪̜̮͙̹n̢͉̝ ͇͉͓̦̼́a̳͖̪̤̱p̖͔͔̟͇͎͠p̱͍̺ę̲͎͈̰̲̤̫a̯͜r̨̮̫̣̘a̩̯͖n̹̦̰͎̣̞̞c̨̦̱͔͎͍͖e̬͓͘ ̤̰̩͙̤̬͙o̵̼̻̬̻͇̮̪f̴ ̡̙̭͓͖̪̤“̸͙̠̼c̳̗͜o͏̼͙͔̮r̞̫̺̞̥̬ru̺̻̯͉̭̻̯p̰̥͓̣̫̙̤͢t̳͍̳̖ͅi̶͈̝͙̼̙̹o̡͔n̙̺̹̖̩͝ͅ”̨̗͖͚̩.̯͓  A few other places where dynamic character composition shows up in Unicode:  Vowel-pointing notation in Arabic and Hebrew. In these languages, words are normally spelled with some of their vowels left out. They then have diacritic notation to indicate the vowels (used in dictionaries, language-teaching materials, children’s books, and such). These diacritics are expressed with combining marks.  A Hebrew example, with niqqud:\tאֶת דַלְתִּי הֵזִיז הֵנִיעַ, קֶטֶב לִשְׁכַּתִּי יָשׁוֹד Normal writing (no niqqud):\tאת דלתי הזיז הניע, קטב לשכתי ישוד Devanagari, the script used to write Hindi, Sanskrit, and many other South Asian languages, expresses certain vowels as combining marks attached to consonant letters. For example, “ह” + “\u200bि” = “हि” (“h” + “i” = “hi”). Korean characters stand for syllables, but they are composed of letters called jamo that stand for the vowels and consonants in the syllable. While there are code points for precomposed Korean syllables, it’s also possible to dynamically compose them by concatenating their jamo. For example, “ᄒ” + “ᅡ” + “ᆫ” = “한” (“h” + “a” + “n” = “han”). Canonical Equivalence In Unicode, precomposed characters exist alongside the dynamic composition system. A consequence of this is that there are multiple ways to express “the same” string—different sequences of code points that result in the same user-perceived characters. For example, as we saw earlier, we can express the character “Á” either as the single code point U+00C1, or as the string of two code points U+0041 U+0301.  Another source of ambiguity is the ordering of multiple diacritics in a single character. Diacritic order matters visually when two diacritics apply to the same side of the base character, e.g. both above: “ǡ” (dot, then macron) is different from “ā̇” (macron, then dot). However, when diacritics apply to different sides of the character, e.g. one above and one below, then the order doesn’t affect rendering. Moreover, a character with multiple diacritics might have one of the diacritics precomposed and others expressed as combining marks.  For example, the Vietnamese letter “ệ” can be expressed in five different ways:  Fully precomposed: U+1EC7 “ệ” Partially precomposed: U+1EB9 “ẹ” + U+0302 “◌̂” Partially precomposed: U+00EA “ê” + U+0323 “◌̣” Fully decomposed: U+0065 “e” + U+0323 “◌̣” + U+0302 “◌̂” Fully decomposed: U+0065 “e” + U+0302 “◌̂” + U+0323 “◌̣” Unicode refers to set of strings like this as “canonically equivalent”. Canonically equivalent strings are supposed to be treated as identical for purposes of searching, sorting, rendering, text selection, and so on. This has implications for how you implement operations on text. For example, if an app has a “find in file” operation and the user searches for “ệ”, it should, by default, find occurrences of any of the five versions of “ệ” above!  Normalization Forms To address the problem of “how to handle canonically equivalent strings”, Unicode defines several normalization forms: ways of converting strings into a canonical form so that they can be compared code-point-by-code-point (or byte-by-byte).  The “NFD” normalization form fully decomposes every character down to its component base and combining marks, taking apart any precomposed code points in the string. It also sorts the combining marks in each character according to their rendered position, so e.g. diacritics that go below the character come before the ones that go above the character. (It doesn’t reorder diacritics in the same rendered position, since their order matters visually, as previously mentioned.)  The “NFC” form, conversely, puts things back together into precomposed code points as much as possible. If an unusual combination of diacritics is called for, there may not be any precomposed code point for it, in which case NFC still precomposes what it can and leaves any remaining combining marks in place (again ordered by rendered position, as in NFD).  There are also forms called NFKD and NFKC. The “K” here refers to compatibility decompositions, which cover characters that are “similar” in some sense but not visually identical. However, I’m not going to cover that here.  Grapheme Clusters As we’ve seen, Unicode contains various cases where a thing that a user thinks of as a single “character” might actually be made up of multiple code points under the hood. Unicode formalizes this using the notion of a grapheme cluster: a string of one or more code points that constitute a single “user-perceived character”.  UAX #29 defines the rules for what, precisely, qualifies as a grapheme cluster. It’s approximately “a base code point followed by any number of combining marks”, but the actual definition is a bit more complicated; it accounts for things like Korean jamo, and emoji ZWJ sequences.  The main thing grapheme clusters are used for is text editing: they’re often the most sensible unit for cursor placement and text selection boundaries. Using grapheme clusters for these purposes ensures that you can’t accidentally chop off some diacritics when you copy-and-paste text, that left/right arrow keys always move the cursor by one visible character, and so on.  Another place where grapheme clusters are useful is in enforcing a string length limit—say, on a database field. While the true, underlying limit might be something like the byte length of the string in UTF-8, you wouldn’t want to enforce that by just truncating bytes. At a minimum, you’d want to “round down” to the nearest code point boundary; but even better, round down to the nearest grapheme cluster boundary. Otherwise, you might be corrupting the last character by cutting off a diacritic, or interrupting a jamo sequence or ZWJ sequence.  And More… There’s much more that could be said about Unicode from a programmer’s perspective! I haven’t gotten into such fun topics as case mapping, collation, compatibility decompositions and confusables, Unicode-aware regexes, or bidirectional text. Nor have I said anything yet about implementation issues—how to efficiently store and look-up data about the sparsely-assigned code points, or how to optimize UTF-8 decoding, string comparison, or NFC normalization. Perhaps I’ll return to some of those things in future posts.  Unicode is a fascinating and complex system. It has a many-to-one mapping between bytes and code points, and on top of that a many-to-one (or, under some circumstances, many-to-many) mapping between code points and “characters”. It has oddball special cases in every corner. But no one ever claimed that representing all written languages was going to be easy, and it’s clear that we’re never going back to the bad old days of a patchwork of incompatible encodings.  Further reading:  The Unicode Standard UTF-8 Everywhere Manifesto Dark corners of Unicode by Eevee ICU (International Components for Unicode)—C/C++/Java libraries implementing many Unicode algorithms and related things Python 3 Unicode Howto Google Noto Fonts—set of fonts intended to cover all assigned code points"""
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience

现在我们可以把词表添加为一个名为 `merges` 的简单字典,以字节对作为键、以表示它的 token 作为值:

In [13]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]): # Sliding Window of size 2 across tokens 
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    # Iterating through ids, if we find (pair), replace it by value idx
    newids = []
    i = 0
    while i < len(ids):
        # If we are not at the very last position AND the pair matches, replace it
        if i < len(ids)-1 and (ids[i], ids[i+1]) == pair:
            newids.append(idx)
            i += 2 # We skip over the now replaced pair
        else:
            newids.append(ids[i])
            i += 1
    return newids


vocab_size = 276    # Our (Arbitrary) Target Vocabulary Size
num_merges = vocab_size - 256
ids = list(tokens)  # We'll work on a copy of tokens from here on (list() makes a deep copy of a list)
merges = {}         # (int, int) -> int, Merge dictionary; Think of this as key: (child1, child2), value: parent/new token


for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get) # Retrieve the most common bigram
    idx = 256 + i
    print(f"Merging {pair}\tinto new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

Merging (101, 32)	into new token 256
Merging (105, 110)	into new token 257
Merging (115, 32)	into new token 258
Merging (116, 104)	into new token 259
Merging (101, 114)	into new token 260
Merging (99, 111)	into new token 261
Merging (116, 32)	into new token 262
Merging (226, 128)	into new token 263
...
Merging (259, 256)	into new token 275


将扩充后的词表与原始词表相比较,可以看到:现在表示同样多的数据所需的 token 更少,因为我们能够用一个相应的 token 来表示更复杂的文本块:

In [14]:
print("Old Length of Token List:", len(tokens))
print("New Length of Token List:", len(ids))
print(f"Compression Ratio: {len(tokens) / len(ids):.2f}x")

Old Length of Token List: 24597
New Length of Token List: 19438
Compression Ratio: 1.27x


仅需 $20$ 次合并和 token 创建,就实现了词表 $1.27$ 倍的压缩。<br><br>
得益于 `merges`,我们既能编码也能解码。tokenizer 已经成为 LLM 与可读文本之间的翻译层。

但是,再说一次,token 深度并不是唯一值得优化的参数。我们从中提取字节对的数据集,应当能很好地代表你想要分词(tokenize)的全部文本范围(至少对于你的预期使用场景而言)。而且,我们已经在 GPT-2 与 GPT-4 的 tokenizer 对比中看到了这种训练数据选择所带来的影响。

### 解码

编码完成后,给定一个取值范围在 $[0;\ \text{vocab\_size}]$ 内的整数 token 序列,对应的文本字符串是什么?

In [15]:
# Decoder Preprocessing: Mapping from token-id to bytes-object for that token
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():   # This needs to run in the order in which we inserted items into merges (use Python >= 3.7)
    vocab[idx] = vocab[p0] + vocab[p1] # Populate at idx (parent integer) with concatenated bytes-object of children p0 and p1, format is {idx: b'p0p1'}

def decode(ids):
    # Given ids (list of integers), return the Python string
    tokens = b"".join(vocab[idx] for idx in ids) # Concatenate all the bytes-objects for each new token-id
    text = tokens.decode("utf-8")                # Decode the bytes into a Python string
    return text

**好的，我们刚才做了什么？**<br>
<br>
字典 `vocab` 在第一步中被设为以字节对象作为值，对应整数键 $0$ 到 $255$。<br>
除了这种 `int -> byte(int)` 的“基础映射”之外，我们现在还想将 `new_token` 作为键，将 `(token1, token2)` 作为值存储起来。<br>
你可以把它想象成把 `(token1, token2) -> new_token` 中 `merges` 的键值关系反过来，变成 `new_token -> (token1, token2)` 中的 `vocab`。<br>
不过，`vocab` 不会以元组的形式存储 `(token1, token2)`。相反，它会存储这两个字节对象的拼接结果，即 `vocab[p0] + vocab[p1]` 或 `byte(token1) + byte(token2)`。

> `vocab` 现在保存了每一个 `token`（不仅是 `new_token`，而是所有 token）的整数到字节的映射，映射目标要么是单个字节，要么是为创建 `new_token` 而合并的两个字节对象的拼接结果。

然后，`decode` 函数会接收一个 token 整数列表，并迭代地在 `vocab` 中按相应的 token 整数进行索引，逐步拼接字节对象以形成一个字符串。<br>
最后，使用 UTF-8 解码将字节对象转换为更易读的内容，理想情况下与原始文本高度接近。<br>
<br>
你可能已经注意到，这里仍然遗留了一个实现问题：

In [16]:
print(decode([97]))  # Works out to be "a"
print(decode([128])) # Works out to be ... flawed?!

a


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte

将 $128_{10}$ 映射为二进制，我们得到 $1000\ 0000_{2}$，根据 [维基百科](https://en.wikipedia.org/wiki/UTF-8#Encoding)，这不是一个有效的 UTF-8 起始字节。<br>
我们提供的是 $1000\ 0000_{2}$，但只有 $0\text{XXX\ XXXX}_{2}$ 或 $110\text{X\ XXXX}_{2}$ 才能开启一个有效的 UTF-8 字节表示。<br>
<br>
**幸运的是，我们可以相当容易地修复这个问题：**

In [17]:
# Decoder Preprocessing: Mapping from token-id to bytes-object for that token
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():   # This needs to run in the order in which we inserted items into merges (use Python >= 3.7)
    vocab[idx] = vocab[p0] + vocab[p1] # Populate at idx (parent integer) with concatenated bytes-object of children p0 and p1

def decode(ids):
    # Given ids (list of integers), return the Python string
    tokens = b"".join(vocab[idx] for idx in ids)     # Concatenate all the bytes-objects for each new token-id
    text = tokens.decode("utf-8", errors="replace")  # Decode the bytes into a Python string
    return text

In [18]:
print(decode([97]))  # Works still
print(decode([128])) # Works now

a
�


**为什么 `errors=replace` 有效？**<br>
<br>
参见 [Python 文档：](https://docs.python.org/3/library/codecs.html#error-handlers) 如果解码器遇到某个无法解码为有效 Unicode 的字节序列，该序列将被替换为一个特殊的 Unicode 替换字符（U+FFFD）。我们基本上是优雅地跳过这个无效的字节序列，喊一声*“我不知道这是什么”*，然后继续。<br>
<br>
**如果我们现在只是跳过这个问题，那它一开始怎么会是个问题呢？我们不是只向编码器输入了 UTF-8 吗？怎么会返回别的东西呢？**<br>
<br>
嗯，是的，我们向编码器输入的是 UTF-8。但实际上，我们解码的不是我们刚刚分词的内容。解码是针对 LLM 输出进行的。<br>
而在那里，如果你遇到一个值为 $128$ 的 LLM 输出符号，说明 LLM 这一侧出了问题，值得投入更多训练来避免。<br>
<br>
**感觉这个 `decode` 的实现太浅了。**<br>
**这个机制怎么能够考虑到 `merges` 中的某些元组本身是由 `new_token` 组成的呢？**<br>
**这难道不会报错吗？**<br>
<br>
这个实现只是乍看之下显得浅。其实并非如此。我们完全按照 `vocab` 被添加到 `new_token` 中的顺序逐步构建 `merges`。这样一来，`new_token` 所代表的元组必定已经存在于 `vocab` 中，随时可供引用。而且，如果你引用一个已经在 `new_token` 中的 `vocab`，你会得到为创建该 `new_token` 而合并的字节对象的拼接结果。字节对象是在 `vocab` 中递归地构建起来的。因此，如果发现某个 `new_token` 代表的是一个由 `new_token` 组成的元组，那么最底层的字节对象会通过递归索引 `vocab` 自动找到。这使得围绕 `vocab[idx] = vocab[p0] + vocab[p1]` 的循环成为查找代表每个 `new_token` 的实际字节对象的一种非常优雅的解决方案。

### 编码

给定一个文本字符串，整数 token 序列 $0$ 到 $\text{vocab\_size} - 1$ 是什么？

In [19]:
def encode(text):
    tokens = list(text.encode("utf-8")) # raw bytes formatted as integer list
    # Now, lookup into merges (historically accurate from top to bottom) and replace the pair with the new token-id recursively
    while True:
        stats = get_stats(tokens) # Count how many times each pair occurs, format: (int, int) -> int (occurrence count)
        pair = min(stats, key=lambda p: merges.get(p, float('inf'))) # Iterating over keys of stats here; Retrieve the pair with the lowest merge index
        if pair not in merges: # This could be because merges doesn't contain any pair that occurs in tokens
            break # We can stop, nothing more to merge here
        idx = merges[pair] # Retrieve new token-representation for mergable pair
        tokens = merge(tokens, pair, idx) # Every pair is replaced with idx, just as we did earlier
    return tokens

对于 `get_stats`，有趣的是我们其实并不真正关心出现次数。更确切地说，`get_stats` 中的键构成了一个值得考虑用于合并的 token 对列表。<br>
从 `merges` 的角度来看，我们希望先执行在 `merges` 中列得最靠前的那些合并。<br>
然后，对于 `pair` 中的任意 `stats`，我们查阅 `merges` 字典。更具体地说，我们查看该 `new_token` 会创建的 `pair`。这样做的目的是找到值最小的 `new_token`。<br>
<br>
例如，假设 `merges` 中的第一个条目是 `(1, 2) -> 256`。<br>
如果这个组合出现在 `tokens` 中，并且被 `get_stats` 认为值得分词，我们就会深入 `merges`。<br>
在那里，我们寻找可归属于某个 pair 的最小可能值（这里是 `256` 对应的 `(1, 2)`），并将其作为 `pair` 的值取回。<br>
正因如此，我们判定 `(1, 2)` 既确实存在于待编码文本中，又被认为最适合分词，因为它的 `new_token` 值最小。<br>
<br>
实现完成后，让我们稍微测试一下：

In [20]:
print(encode("hello world!"))
print(encode("h"))

[104, 101, 108, 108, 111, 32, 119, 266, 108, 100, 33]


ValueError: min() arg is an empty sequence

如果只有一个字符或一个空字符串，我们就无法运行 `stats = get_stats(tokens)`，因为找不到任何 pair。<br>
<br>
*让我们修复这个问题：*

In [21]:
def encode(text):
    tokens = list(text.encode("utf-8")) # raw bytes formatted as integer list
    # Now, lookup into merges (historically accurate from top to bottom) and replace the pair with the new token-id recursively
    while len(tokens) > 1:
        stats = get_stats(tokens) # Count how many times each pair occurs, format: (int, int) -> int (occurrence count)
        pair = min(stats, key=lambda p: merges.get(p, float('inf'))) # Iterating over keys of stats here; Retrieve the pair with the lowest merge index
        if pair not in merges: # This could be because merges doesn't contain any pair that occurs in tokens
            break # We can stop, nothing more to merge here
        idx = merges[pair] # Retrieve new token-representation for mergable pair
        tokens = merge(tokens, pair, idx) # Every pair is replaced with idx, just as we did earlier
    return tokens

print(encode("hello world!"))
print(encode("h"))
print(encode(""))

[104, 101, 108, 108, 111, 32, 119, 266, 108, 100, 33]
[104]
[]


最后，让我们解码一些已编码的内容：

In [22]:
print(decode(encode("hello world!"))) # Works

# We know this text already, the tokenizer was trained on it
text2 = decode(encode(text))
print('Training Text is Reconstructable: ', text2 == text) # Works

# We don't know this text, it's not in the training data
valtext = "Many common characters, including numerals, punctuation, and other symbols, are unified within the standard and are not treated as specific to any given writing system. Unicode encodes thousands of emoji, with the continued development thereof conducted by the Consortium as a part of the standard.[4] Moreover, the widespread adoption of Unicode was in large part responsible for the initial popularization of emoji outside of Japan. Unicode is ultimately capable of encoding more than 1.1 million characters."
valtext2 = decode(encode(valtext))
print('Unseen Text is Reconstructable:   ', valtext2 == valtext) # Works

hello world!
Training Text is Reconstructable:  True
Unseen Text is Reconstructable:    True


**这个 `decode(encode(x))` 对每个 `x` 都能得到 `x` 吗？**<br>
<br>
不，并不总是如此。在处理纯 UTF-8 文本时，我们可以达到这种编码质量。<br>然而，在处理混合文本时，我们无法保证 $1:1$ 的重建。<br>
但无论如何，我们刚刚朝着比字符级分词更高效的分词方式迈出了一大步。

## 走向 SOTA：实战中的 Tokenizer

### GPT Tokenizer

根据 [\[Radford et al., 2019\]](https://insightcivic.s3.us-east-1.amazonaws.com/language-models.pdf)，GPT-2 是首个推动使用 byte-pair encoding [\[Sennrich et al., 2015\]](https://arxiv.org/abs/1508.07909) 进行分词的 GPT 版本。<br>
有趣的是，GPT-2 在很大程度上正是像我们这样实现 BPE 的。但随后论文的做法就有所不同了。<br><br>
设想一个在数据集中出现频率非常高的词。每次它出现时，周围可能伴随不同的词/字符。<br>
取决于 tokenizer 如何处理这一情况，可能会发生几种情形：
- 如果某些部分经常共同出现，tokenizer 可能会决定将该词与其周围上下文的部分一起拆分或合并。
- 如果该词的某些部分会随上下文而变化，tokenizer 可能会为同一个词的不同部分创建多个 token。
- tokenizer 甚至可能开始将该词的某些部分与上下文中其他频繁出现的前导或尾随字符关联起来。

正如论文所指出的，这种行为是次优的。它所产生的 token 反映的是统计频率而非真正的上下文含义，导致出现一些看似数量充足、但实际在语义上并不相关的聚类。
<br>
为解决这一问题，某些类型的字符被强制规定永远不能被合并。<br>
实际的 tokenizer 实现请参见 [GitHub - OpenAI/gpt-2/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)。<br>
具体而言，请参阅第 $53$ 行。其中包含一个 regex 模式，用于排除某些字符使之不被合并在一起。<br>

In [3]:
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""") # |: OR operator

# 's, 't, 're, 've, 'm, 'll, 'd are all contractions (curiously case-sensitive) to be singled out
# " ?\p{L}+" matches any sequence of letters with an optional leading space
# " ?\p{N}+" matches any sequence of digits with an optional leading space
# " ?[^\s\p{L}\p{N}]+" matches any sequence of non-whitespace, non-letter, non-digit characters with an optional leading space
# "\s+(?!\S)" matches any sequence of whitespace characters that are not followed by a non-whitespace character
# "\s+" matches any sequence of whitespace characters

print(re.findall(gpt2pat, "Hello world how are you? I've heard you're      4.543 billion years old??!")) # Works, """ ?\p{L}+""" matches "Hello", """ ?\p{L}+""" matches "world" etc.

['Hello', ' world', ' how', ' are', ' you', '?', ' I', "'ve", ' heard', ' you', "'re", '     ', ' 4', '.', '543', ' billion', ' years', ' old', '??!']


GPT-2 并不直接按原样对输入进行编码，而是将其拆分为一个由符合 regex 规则的输入文本块组成的列表。<br>
列表中的各个条目会被分别处理，然后将它们的 token 表示拼接在一起。<br>

> 实际上，你只能在符合 regex 规则的块的边界内寻找 token，而不能跨越这些边界。

有趣的是，`r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""` 其实相当次优。<br>
想想全大写的 `"SHOULD'VE TESTED THAT"`。<br>
上面针对 `'ve` 的规则并不能匹配，因为它们是大小写敏感的。

In [24]:
print(re.findall(gpt2pat, "SHOULD'VE TESTED THAT"))
print()

example = """for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)
"""
print(re.findall(gpt2pat, example))

['SHOULD', "'", 'VE', ' TESTED', ' THAT']

['for', ' i', ' in', ' range', '(', '1', ',', ' 101', '):', '\n   ', ' if', ' i', ' %', ' 3', ' ==', ' 0', ' and', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'FizzBuzz', '")', '\n   ', ' elif', ' i', ' %', ' 3', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Fizz', '")', '\n   ', ' elif', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Buzz', '")', '\n   ', ' else', ':', '\n       ', ' print', '(', 'i', ')', '\n']


人们可能会认为现在可以直接在这些输入文本块上运行 BPE，但事实并非如此。<br>
不过，令人好奇的是，请看一下代码示例所得到的符合 regex 规则的块的输出，并将其与 [TikTokenizer](https://tiktokenizer.vercel.app) 的输出进行对比：

![](./img/Tiktoken_Vercel_1.png)

**有些地方对不上。**<br>
OpenAI 似乎在符合 regex 规则的块与准备好进行 BPE 分词的文本片段之间，加入了一个额外的分块与切片步骤。<br>
原因是什么？目前我们只能推测。

### OpenAI TikToken

TikToken 是 [OpenAI 官方的 tokenizer 实现](https://github.com/openai/tiktoken)。<br>
它提供了对 GPT-2、GPT-3、GPT-4 和 GPT-5 所用 tokenizer 的接口：

In [25]:
# GPT-2 Tokenizer
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    Hello World?!!")) # whitespace remains unmerged, as seen on TikTokenizer.vercel.app

# GPT-4 Tokenizer
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    Hello World?!!"))

[220, 220, 220, 18435, 2159, 30, 3228]
[262, 22691, 4435, 30, 3001]


[参见 OpenAI TikToken 仓库中的这个文件](https://github.com/openai/tiktoken/blob/main/tiktoken_ext/openai_public.py)。<br><br>
对于 GPT-2 的分词，我们可以看到 `r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""` 在功能上等价于 [GitHub - OpenAI/gpt-2/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)<br><br>
对于 GPT-4 的分词，我们还可以看到该模式被改进为 `r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""`。<br>
<br>
下面是这个 regex 模式的解读：

In [ ]:
gpt4pat = re.compile(r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+""") 

# | is the logical OR operator

# (?i:[sdmt]|ll|ve|re)       matches 's, 'd, 'm, 't, 'll, 've, 're *non case-sensitive* now
# [^\r\n\p{L}\p{N}]?+\p{L}+  matches letter sequences with optional leading non-letter, non-digit characters
# \p{N}{1,3}                 matches digit sequences of length 1 to 3 (interesting! prevents long number tokens)
#  ?[^\s\p{L}\p{N}]++[\r\n]* matches optional spaced followed by one or more non-space, non-letter, non-number characters, then allows for however many newlines
# \s*[\r\n]                  matches none or more whitespace characters, followed by a single newline character
# \s+(?!\S)                  matches one or more whitespaces that are not followed by a non-whitespace character
# \s+                        matches one or more whitespace characters (no negative lookahead)

print(re.findall(gpt4pat, "P. Sherman, 42 Wallaby Way, Sydney")) # Works, """ ?\p{L}+""" matches "Hello", """ ?\p{L}+""" matches "world" etc.

['P', '.', ' Sherman', ',', ' ', '42', ' Wallaby', ' Way', ',', ' Sydney']


让我们实际回到 [GitHub - OpenAI/gpt-2/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)。<br>
我们可以看到 tokenizer 是从 `encoder.json` 文件和 `vocab.bpe` 文件中获取的。<br>
- `encoder.json` 保存了 token 到整数的映射。
- `vocab.bpe` 保存了 byte-pair encoding。

In [ ]:
# Retrieve the GPT-2 Tokenizer just like TikToken does
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json

In [27]:
# This is equivalent to our `vocab`, Format is equal, just inverted: {idx: b'p0p1'} for us, {b'p0p1': idx} for them
with open('encoder.json', 'r') as f:
    encoder = json.load(f)

# This is equivalent to our `merges`, Format is equal for bpe_merges: {(child1, child2): idx}
with open('vocab.bpe', 'r', encoding="utf-8") as f:
    bpe_data = f.read()

bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]

In [28]:
print("Vocab Format is {b'p0p1': idx}:           ", list(encoder.items())[:5])
print("Merges Format is {(child1, child2): idx}: ", bpe_merges[:5])

Vocab Format is {b'p0p1': idx}:            [('!', 0), ('"', 1), ('#', 2), ('$', 3), ('%', 4)]
Merges Format is {(child1, child2): idx}:  [('Ġ', 't'), ('Ġ', 'a'), ('h', 'e'), ('i', 'n'), ('r', 'e')]


简而言之，我们实现了一个可训练的 BPE tokenizer，OpenAI 也同样实现了。<br><br>
然而，他们只公开了训练后的结果，而没有公开我们自己实现的那种训练代码——有趣的是，我们实现的训练代码与 OpenAI 产生的文件格式也是兼容的。<br>
尽管如此，我们获得的分词结果与 OpenAI 所获得的结果并不相同，因为他们在架构上做出了一个决定：在伪公开的 BPE tokenizer 之上再加了一个第二层、闭源、非公开的编码器。

### 特殊 Token

除了通过 regex 分块来引导 token 的派生之外，我们还可以引入特殊 token 并将其添加到词表中。<br>

In [29]:
print(len(encoder)) # 256 raw byte tokens + 50,000 merges + --> 1 special token <--
print(list(encoder.items())[len(encoder)-1])

50257
('<|endoftext|>', 50256)


> 特殊的 `<|endoftext|>` 被插入到训练集中，用于标记一个（上下文相关的）文档的结束和另一个文档的开始。模型应当学会插入一个概念性的“断点”，以免把连续但可能并不相关的文档内容混在一起。

![](./img/Tiktoken_Vercel_2.png)

可以把特殊 token 看作是“注入”的 token，不属于 BPE 所称的词表的一部分。<br><br>
特殊 token 可以被注入然后进行训练。例如，某个特殊 token 的出现可能会触发一个网络搜索流程，以丰富 LLM 的上下文。<br>
这对于微调（finetuning）而言也是一种关键架构，例如用于虚拟独白的 `<|im_start|>`<br>
或者根据 [\[Bavarian et al., 2022\]](https://arxiv.org/abs/2207.14255) 的 `<|fim_prefix|>`、`<|fim_middle|>`、`<|fim_suffix|>` *(这实际上在 GPT-4 中被使用)*。<br><br>
你完全可以去 fork Tiktoken 库，并通过（请注意保持索引一致）自定义特殊 token 来扩展它。Tiktoken 会为你处理其余的一切。


### Sentencepiece

开源的 [Sentencepiece](https://github.com/google/sentencepiece) 库可以完成 BPE 分词（以及其他算法）的训练和推理。<br>

> Llama 和 Mistral 模型依赖 Sentencepiece 进行分词。

在使用 tiktoken 时，我们拿到字符串数据，将其解码为 UTF-8，然后从那里对字节进行分词。<br>
Sentencepiece 直接在 Unicode 码点（codepoint）上运行 BPE，而不是在 UTF-8 字节表示上运行。<br>
它有一个 `character_coverage` 选项，用于处理出现次数很少的码点：要么将它们映射到一个 UNK token，要么在启用了 `byte_fallback` 时，<br>
用 UTF-8 编码它们，然后对原始字节进行分词。<br>
<br>
**tiktoken 在概念上更干净，而 Sentencepiece 更高效。**

In [30]:
import sentencepiece as spm

In [31]:
# Sentencepiece likes to work with files, so let's write our training data to a file:
with open("toy.txt", "w", encoding="utf-8") as f:
    # From the SentencePiece README (https://github.com/google/sentencepiece)
    f.write("SentencePiece is an unsupervised text tokenizer and detokenizer mainly for Neural Network-based text generation systems where the vocabulary size is predetermined prior to the neural model training. SentencePiece implements subword units (e.g., byte-pair-encoding (BPE) [Sennrich et al.]) and unigram language model [Kudo.]) with the extension of direct training from raw sentences. SentencePiece allows us to make a purely end-to-end system that does not depend on language-specific pre/postprocessing. This is not an official Google product.")

In [32]:
options = dict(
  # input
  input="toy.txt",
  input_format="text",
  # output
  model_prefix="tok400", # output filename prefix
  # algorithm spec
  model_type="bpe", # BPE algorithm
  vocab_size=400,
  # normalization
  normalization_rule_name="identity", # ew, turn off normalization
  remove_extra_whitespaces=False,
  input_sentence_size=200000000, # max number of training sentences
  max_sentence_length=4192, # max number of bytes per sentence
  seed_sentencepiece_size=1000000,
  shuffle_input_sentence=True,
  # rare word treatment
  character_coverage=0.99995,
  byte_fallback=True,
  # merge rules (a different way to approach what tiktoken did through regex)
  split_digits=True,
  split_by_unicode_script=True,
  split_by_whitespace=True,
  split_by_number=True,
  max_sentencepiece_length=16,
  add_dummy_prefix=True,
  allow_whitespace_only_pieces=True,
  # special hard-coded tokens
  unk_id=0, # the UNK token MUST exist
  bos_id=1, # the others are optional, set to -1 to turn off
  eos_id=2,
  pad_id=-1,
  # systems
  num_threads=os.cpu_count(), # use ~all system resources
)

spm.SentencePieceTrainer.train(**options)

In [33]:
sp = spm.SentencePieceProcessor()
sp.load("tok400.model")

# Inspect vocabulary off the model file
vocab = [[sp.id_to_piece(idx), idx] for idx in range(sp.get_piece_size())]
vocab

[['<unk>', 0],
 ['<s>', 1],
 ['</s>', 2],
 ['<0x00>', 3],
 ['<0x01>', 4],
 ['<0x02>', 5],
 ['<0x03>', 6],
 ['<0x04>', 7],
...
 ['T', 399]]


In [34]:
# Now, with the vocabulary sorted out, we can encode and decode text
ids = sp.encode("hello 안녕하세요")
print(ids) # This is the tokenized representation of the text

[359, 376, 360, 370, 370, 364, 359, 239, 152, 139, 238, 136, 152, 240, 152, 155, 239, 135, 187, 239, 157, 151]


In [35]:
print([sp.id_to_piece(idx) for idx in ids]) # This is the representation of what the token ids refer to

['▁', 'h', 'e', 'l', 'l', 'o', '▁', '<0xEC>', '<0x95>', '<0x88>', '<0xEB>', '<0x85>', '<0x95>', '<0xED>', '<0x95>', '<0x98>', '<0xEC>', '<0x84>', '<0xB8>', '<0xEC>', '<0x9A>', '<0x94>']


在上面的例子中，Sentencepiece 遇到了 `vocab` 未覆盖的输入。<br>
如果不是因为 `byte_fallback=True`，这将会是个问题。<br>
设置了该标志后，Sentencepiece 会将输入编码为 UTF-8，并将原始字节作为回退方案进行分词。

> 如果不是因为有 `byte_fallback=True`，我们会得到 `['_', 'h', 'e', 'l', 'l', 'o', '_', '<unk>']`，因为我们从未在训练数据中真正见过这些韩文字符，从未将它们映射到 token，这会导致它们被无情地替换为 `<unk>` token。

顺便说一下，为什么前面会多出一个 `'_'` token？<br>
这是由 `add_dummy_prefix=True` 造成的，目的是让句子中间的 `_hello` 和句子开头的 `hello` 以相同方式编码。

我们刚刚走过的设置与 Llama 的训练方式非常接近。<br>
实际上，如果你想要 Llama 的*精确*设置，请参考 [这个 issue](https://github.com/google/sentencepiece/issues/121)。

---

## 练习时间

你可以在配套课程的 [代码仓库](https://github.com/karpathy/minbpe/blob/master/exercise.md) 中找到一个实现 BPE tokenizer 的练习。<br>在这里解决这个练习并不适合本 notebook。<br><br>

---

## 回过头来：vocab_size

我们在本章开头已经讨论过这个问题，但在[上一章](../N007%20-%20GPT%20From%20Scratch/N007%20-%20GPT.ipynb)中，我们用 PyTorch 实现了 [`gpt.py`](../N007%20-%20GPT%20From%20Scratch/gpt.py)。<br>
我们的 `vocab_size` 是 $65$。我们用一个 $65$ 维的向量来表示每个 token。不过，这仍然是*字符级分词*。<br>
显而易见，随着 `vocab_size` 增长，这种方法的扩展性很差，因为在 $65$ 个向量表示的每一个中，<br>
我们都记录了 $65$ 个 token 中每一个紧跟在当前 token 之后的可能性。

> 这个分布越大，每个向量在表达下一个位置应该偏好哪个独特 token 方面就越缺乏表达力。

In [ ]:
torch.manual_seed(1337) # for reproducibility

# Not really an LM at this stage, but we will get there...
class BigramLM(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Embedding the vocabulary
        # Every one of the vocab_size tokens is represented by a vector of size vocab_size
        self.embed = nn.Embedding(vocab_size, vocab_size) # 65 unique 65-dim vectors

    def forward(self, idx, targets):
        # idx is of shape (batch_size, block_size)
        # targets is of shape (batch_size, block_size)
        logits = self.embed(idx)
        return logits # Embed the input indices, shape is now (batch_size, block_size, vocab_size) (B, T, C)

如果我们想使用一个现有模型并扩展它的词表大小呢？如果要添加一个特殊 token 呢？<br>
这两者都可以做到，但模型的嵌入矩阵必须重新训练。不过这件事可以做得相当好。<br>

## 结论

说实话，对 token 词表进行建模、重塑和微调是一个非常有趣的研究领域，参见 [\[Mu et al., 2023\]](https://arxiv.org/abs/2304.08467)。<br>
除此之外，词表与 tokenization 的问题越来越不再仅仅是建模语言的工具。<br>
例如 [\[Esser et al., 2020\]](https://arxiv.org/abs/2012.09841) 这类论文研究了如何将图像作为额外的模态输入到 LLM 中。<br>

> 看起来这一领域正在经历一种趋同：transformer 不可触碰，但 tokenization 可以被扩展。可以说是“假装这张图片也是文本”。<br>
> 这正是 OpenAI SORA 所做的，将图像输入高效地切分成 token，并在下游用 transformer 架构进行处理。

好了，作为总结，让我们试着为前面提出的这些问题找到答案：

- **为什么 LLM 不会拼写？**
    - 字符被切分成了大小不一的 token；提示 `How many letters 'l' are in ".DefaultCellStyle"` 会得到各种错误的结果。
- **为什么 LLM 无法完成像反转字符串这样的简单文本操作？**
    - 同样地，字符被切分成了大小不一的 token；这会让 ChatGPT 栽跟头，但正因如此，反转 `.D e f a u l t C e l l S t y l e` 反而能奏效。
- **为什么 LLM 在非英语语言上表现更差？**
    - 训练数据中代表性较低，导致非英语字符的 tokenization 更少（甚至没有）。这会撑爆 Attention 缓冲区。糟糕。
- **为什么 LLM 无法正确完成简单的算术运算？**
    - 数字的 tokenization 是其根本原因。或者更贴切地说，[Integer tokenization is insane](https://www.beren.io/2023-02-04-Integer-tokenization-is-insane/)
- **为什么输入像 `<|endoftext|>` 这样的特殊字符会让生成停滞？**
    - 它是那些特殊 token 之一。因此它会被非常特定地解释。不要把它当作一个 token，而要当作一条命令
- **为什么 LLM 会因为“尾部空格”而出毛病？**
    - token（例如 GPT 的）通常遵循 `(space)(something)` 的模式，因此添加一个空格会让 GPT 试图想出某种与这个已有空格相匹配的内容，而这很罕见，因此会“污染”上下文。
- **为什么 LLM 在遇到单词中出现大写字母时会崩溃？**
    - LLM 从未见过这种情况；它会困惑，并很快转向预测词尾 token
- **为什么通过 YAML 与 LLM 协作可能比通过 JSON 更好？**
    - YAML 包含的特殊字符更少，因此更不容易让 tokenizer 出错
- **为什么 LLM 并不真正意味着端到端的语言建模？**
    - 我们需要 Tokenize 来形成统一且可扩展的文本表示基础
- **[`SolidGoldMagikarp`](https://www.lesswrong.com/posts/aPeJE8bSo6rAFoLqg/solidgoldmagikarp-plus-prompt-generation) 到底是怎么回事？**
    - 这彻底搞坏了像 ChatGPT 这样的 LLM，本质上相当于越狱。看起来 reddit 用户 [SolidGoldMagikarp](https://reddit.com/u/SolidGoldMagikarp) 发帖太频繁，以至于 tokenizer 在训练中认识它，但 LLM 并不认识


> 谁能摆脱 tokenization，谁就能获得永恒的荣耀。


## 参考文献
这里列出的是 Andrej Karpathy 课程之外的参考资料，包括相关论文、博客文章和其他资源。

- [**回顾：字符级 Tokenization**](#回顾字符级-tokenization)
- [**Tokenization 的陷阱**](#分词的陷阱)
  - [Language Models are Unsupervised Multitask Learners \[Radford et al., 2019\]](https://insightcivic.s3.us-east-1.amazonaws.com/language-models.pdf)
  - [Tiktokenizer.vercel.app](https://tiktokenizer.vercel.app/)
- [**Tokenizer 的思想**](#tokenizer-的思想)
  - [**Unicode**](#unicode)
    - [Nathan Reed - Programmer's Intro to Unicode](https://www.reedbeta.com/blog/programmers-intro-to-unicode/)
    - [UTF-8 Everywhere Manifesto](https://utf8everywhere.org/)
    - [MEGABYTE: Predicting Million-byte Sequences with Multiscale Transformers \[Yu et al., 2023\]](https://arxiv.org/abs/2305.07185)
- [**Byte-Pair Encoding**](#字节对编码)
  - [**解码**](#解码)
  - [**编码**](#编码)
- [**迈向 SOTA：实战中的 Tokenizer**](#走向-sota实战中的-tokenizer)
  - [**GPT Tokenizer**](#gpt-tokenizer)
    - [Language Models are Unsupervised Multitask Learners \[Radford et al., 2019\]](https://insightcivic.s3.us-east-1.amazonaws.com/language-models.pdf)
    - [Neural Machine Translation of Rare Words with Subword Units \[Sennrich et al., 2015\]](https://arxiv.org/abs/1508.07909)
    - [GitHub.com/openai/GPT-2](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
  - [**OpenAI TikToken**](#openai-tiktoken)
    - [GitHub.com/openai/tiktoken](https://github.com/openai/tiktoken)
  - [**特殊 Token**](#特殊-token)
    - [Efficient Training of Language Models to Fill in the Middle \[Bavarian et al., 2022\]](https://arxiv.org/abs/2207.14255)
  - [**Sentencepiece**](#sentencepiece)
    - [GitHub.com/google/sentencepiece](https://github.com/google/sentencepiece)
    - [GitHub.com/google/sentencepiece Issue #121](https://github.com/google/sentencepiece/issues/121)
- [**练习时间**](#练习时间)
    - [GitHub.com/karpathy/minbpe](https://github.com/karpathy/minbpe)
- [**回过头来：vocab_size**](#回过头来vocab_size)
- [**结论**](#结论)
    - [Learning to Compress Prompts with Gist Tokens \[Mu et al., 2023\]](https://arxiv.org/abs/2304.08467)
    - [Taming Transformers for High-Resolution Image Synthesis \[Esser et al., 2020\]](https://arxiv.org/abs/2012.09841)
    - [Beren Millidge: Integer tokenization is insane](https://www.beren.io/2023-02-04-Integer-tokenization-is-insane/)
    - [Jessica Rumbelow: SolidGoldMagikarp (plus, prompt generation)](https://www.lesswrong.com/posts/aPeJE8bSo6rAFoLqg/solidgoldmagikarp-plus-prompt-generation)

<center>Notebook by <a href="https://github.com/mk2112" target="_blank">mk2112</a>.</center>